# Parda — web app M4b, step A: EasyOCR to ONNX (the second OCR engine in the browser)

Exports EasyOCR's text detector (CRAFT) and its three recognizers (`en`, `hi+en`, `kn+en`) to ONNX, then proves the
converted networks give **exactly the same OCR** as PyTorch EasyOCR on real pages (scans and phone photos), by swapping
only the networks inside the real EasyOCR. Then records inputs/outputs + EasyOCR's source for the TypeScript port and
uploads the ONNX files to your HF repo.

**Settings:** Accelerator **None (CPU)** · Internet **On**
**Inputs:** Your Work → **Parda** (Phase 1) · Datasets tab → `celeba-dataset` · **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`
Run all cells (~40 min). Then send cells 4–5 output and attach **`easyocr_goldens.zip`** in the chat.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to make the phone-photo pages), Tesseract + EasyOCR, ONNX tools
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh
!pip install -q onnx onnxruntime
!python -c "import easyocr, torch, onnx, onnxruntime; print('easyocr', easyocr.__version__, '| torch', torch.__version__, '| onnxruntime', onnxruntime.__version__)"


In [ ]:
# 3. Real pages: 30 seen benchmark pages (scans, real faces) + 30 unseen phone photos, 10 per language each
import json, shutil
from parda.ocr.run_ocr import find_data
from parda.vision.faces import find_faces_dir
from parda.pipeline.evaluate_e2e import load_pages
W = "/kaggle/working"
DATA = find_data()
FACES = find_faces_dir("/kaggle/input", exclude=[DATA])
assert DATA and FACES, "attach the Parda (Phase 1) notebook and the celeba-dataset"
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!python -m parda.synth.ood --n 30 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
pages = []
for kind, ps in (("scan", load_pages(DATA, f"{W}/yolo_bench")), ("photo", load_pages(f"{W}/ood"))):
    per = {}
    for p in ps:
        if len(per.setdefault(p["rec"]["lang"], [])) < 10:
            per[p["rec"]["lang"]].append({"id": f"{kind}{p['id']}", "path": p["path"], "lang": p["rec"]["lang"]})
    pages += [x for v in per.values() for x in v]
with open(f"{W}/pages.jsonl", "w") as f:
    for p in pages:
        f.write(json.dumps(p) + "\n")
print(len(pages), "pages:", {k: sum(p["lang"] == k for p in pages) for k in ("en", "hi-en", "kn-en")})


In [ ]:
# 4. EXPORT: CRAFT detector + 3 recognizers to ONNX (~5 min)
!python tools/easyocr_onnx.py export --pages {W}/pages.jsonl --out {W}/easyocr_onnx 2>&1 | grep -v -i warning | tail -12


In [ ]:
# 5. PARITY: PyTorch EasyOCR vs the same EasyOCR with ONNX networks, on 20 pages per language (scans + photos) (~25 min)
pages_mix = [p for lang in ("en", "hi-en", "kn-en") for kind in ("scan", "photo") for p in pages
             if p["lang"] == lang and p["id"].startswith(kind)][:]
with open(f"{W}/pages_check.jsonl", "w") as f:
    for p in pages_mix:
        f.write(json.dumps(p) + "\n")
!python tools/easyocr_onnx.py check --onnx {W}/easyocr_onnx --pages {W}/pages_check.jsonl --n 20 2>&1 | grep -v -i warning | tail -5


In [ ]:
# 6. RECORD for the TypeScript port: 2 pages per language (1 scan + 1 photo) + EasyOCR's source (~3 min)
rec_pages = []
for lang in ("en", "hi-en", "kn-en"):
    for kind in ("scan", "photo"):
        rec_pages.append(next(p for p in pages if p["lang"] == lang and p["id"].startswith(kind)))
with open(f"{W}/pages_record.jsonl", "w") as f:
    for p in rec_pages:
        f.write(json.dumps(p) + "\n")
!python tools/easyocr_onnx.py record --onnx {W}/easyocr_onnx --pages {W}/pages_record.jsonl --n 2 --out {W}/easyocr_goldens 2>&1 | grep -v -i warning | tail -8
shutil.copy(f"{W}/easyocr_onnx/easyocr_onnx.json", f"{W}/easyocr_goldens/easyocr_onnx.json")
shutil.copy(f"{W}/easyocr_onnx/parity.json", f"{W}/easyocr_goldens/parity.json")
shutil.make_archive(f"{W}/easyocr_goldens", "zip", f"{W}/easyocr_goldens")
print(f"easyocr_goldens.zip: {os.path.getsize(f'{W}/easyocr_goldens.zip') / 2**20:.1f} MB")


In [ ]:
# 7. Save the ONNX networks to your HF repo (the web app loads them from there)
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
HF_USER = api.whoami()["name"]
api.upload_folder(repo_id=f"{HF_USER}/parda-onnx-v1", folder_path=f"{W}/easyocr_onnx", path_in_repo="easyocr",
                  commit_message="EasyOCR (CRAFT + en / hi+en / kn+en recognizers) as ONNX")
print(json.load(open(f"{W}/easyocr_onnx/easyocr_onnx.json"))["mb"])
